In [32]:
from pprep.pipeline import prepare_dataset_from_yaml

In [33]:
data = prepare_dataset_from_yaml("adult")

X_train = data['X_train']
X_test = data['X_test']
X_val = data['X_val']

y_train = data['y_train'].to_numpy().ravel()
y_test = data['y_test'].to_numpy().ravel()
y_val = data['y_val'].to_numpy().ravel()

s_train = data['s_train'].to_numpy().ravel()
s_test = data['s_test'].to_numpy().ravel()
s_val = data['s_val'].to_numpy().ravel()


================ DATASET CHECK ================

Samples: 45222
Features: 12

Target (y):
income
0         34014
1         11208
Name: count, dtype: int64

Sensitive (s):
{'sex': {1: 30527, 0: 14695}}

NaNs check:
X NaNs: 0
y NaNs: 0
s NaNs: 0

Sanity check:
Target unique values: [0 1]
Sensitive unique values: {'sex': [1, 0]}




In [ ]:
import numpy as np
import torch
from scipy.spatial.distance import pdist, squareform

In [ ]:
import numpy as np
import torch
from scipy.spatial.distance import pdist, squareform


class MahalanobisLaplacianBuilder:
    def __init__(
        self,
        theta=1.0,
        tau_quantile=0.25,
        laplacian_type="unnormalized",
        eps=1e-8,
        dtype=torch.float32,
        device="cpu",
    ):
        self.theta = theta
        self.tau_quantile = tau_quantile
        self.laplacian_type = laplacian_type
        self.eps = eps
        self.dtype = dtype
        self.device = device

    def _to_numpy(self, X):
        if isinstance(X, torch.Tensor):
            return X.detach().cpu().numpy()

        if hasattr(X, "to_numpy"):
            return X.to_numpy()

        return np.asarray(X)

    def _build_W(self, X):
        X = self._to_numpy(X).astype(np.float64)

        cov = np.cov(X, rowvar=False)
        cov = cov + self.eps * np.eye(cov.shape[0])

        inv_cov = np.linalg.inv(cov)

        dist_vector = pdist(X, metric="mahalanobis", VI=inv_cov)

        tau = np.quantile(dist_vector[dist_vector > 0], self.tau_quantile)

        W_vector = np.exp(-self.theta * dist_vector**2)
        W_vector[dist_vector > tau] = 0.0

        W = squareform(W_vector)

        # Garante simetria
        W = 0.5 * (W + W.T)

        np.fill_diagonal(W, 0.0)

        return W

    def _unnormalized_laplacian(self, W):
        degree = W.sum(axis=1)

        L = -W.copy()
        np.fill_diagonal(L, degree)

        return L

    def _normalized_random_walk_laplacian(self, W):
        degree = W.sum(axis=1)

        inv_sqrt_degree = 1.0 / np.sqrt(degree + self.eps)

        W_tilde = (
            inv_sqrt_degree[:, None]
            * W
            * inv_sqrt_degree[None, :]
        )

        degree_tilde = W_tilde.sum(axis=1)

        L = -W_tilde.copy()

        # L = I - D_tilde^{-1} W_tilde
        # Então fora da diagonal: - W_tilde_ij / degree_tilde_i
        L = L / (degree_tilde[:, None] + self.eps)

        np.fill_diagonal(L, 1.0)

        return L

    def build(self, X, return_artifacts=False):
        W = self._build_W(X)

        if self.laplacian_type == "unnormalized":
            L = self._unnormalized_laplacian(W)

        elif self.laplacian_type == "normalized_random_walk":
            L = self._normalized_random_walk_laplacian(W)

        else:
            raise ValueError(
                "laplacian_type deve ser 'unnormalized' "
                "ou 'normalized_random_walk'."
            )

        L_torch = torch.tensor(
            L,
            dtype=self.dtype,
            device=self.device
        )

        if return_artifacts:
            return {
                "W": W,
                "L": L_torch,
                "laplacian_type": self.laplacian_type,
            }

        return L_torch

In [62]:
builder = MahalanobisLaplacianBuilder(
    theta=1.0,
    tau_quantile=0.25,
    device="cpu"
)

builder.build(X_train)

tensor([[ 2.6163e+00, -0.0000e+00, -4.3094e-14,  ..., -5.5358e-16,
         -6.2967e-16, -0.0000e+00],
        [-0.0000e+00,  1.8794e+00, -0.0000e+00,  ..., -0.0000e+00,
         -0.0000e+00, -0.0000e+00],
        [-4.3094e-14, -0.0000e+00,  1.1433e+01,  ..., -1.0654e-14,
         -2.6567e-06, -0.0000e+00],
        ...,
        [-5.5358e-16, -0.0000e+00, -1.0654e-14,  ...,  9.3022e-01,
         -1.4813e-14, -0.0000e+00],
        [-6.2967e-16, -0.0000e+00, -2.6567e-06,  ..., -1.4813e-14,
          1.7588e+01, -0.0000e+00],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00,  ..., -0.0000e+00,
         -0.0000e+00,  8.4793e-14]])

In [ ]:
import torch
from .metric import Metric


class LaplacianFairnessMetric(Metric):
    name = "lap"
    direction = "min"
    type = "fairness"

    def __init__(self, L, normalize=True, symmetrize=True):
        """
        L : torch.Tensor
            Matriz laplaciana com shape (n, n).

        normalize : bool
            Se True, divide a métrica por n.

        symmetrize : bool
            Se True, usa (L + L.T) / 2.
            Útil para Laplacianos não simétricos, como random walk.
        """
        self.L = L.float()
        self.normalize = normalize
        self.symmetrize = symmetrize

    def __call__(self, y_true, y_pred, sensitive_attr=None):
        """
        y_true:
            Não é usado diretamente, mas mantido para seguir a interface Metric.

        y_pred:
            Pode ser:
                - tensor (n,) com predições escalares;
                - tensor (n, k) com probabilidades ou logits.
        """

        F = y_pred.float()

        if F.ndim == 1:
            F = F.view(-1, 1)

        L = self.L.to(F.device)

        if self.symmetrize:
            L = 0.5 * (L + L.T)

        if L.shape[0] != F.shape[0]:
            raise ValueError(
                f"L tem shape {L.shape}, mas y_pred tem shape {F.shape}. "
                "O Laplaciano precisa ter sido gerado sobre o mesmo conjunto."
            )

        value = torch.sum((L @ F) * F)

        if self.normalize:
            value = value / F.shape[0]

        return value.item()

In [ ]:
X = X_train.to_numpy()

cov = np.cov(X, rowvar=False)

inv = np.linalg.inv(cov)

In [58]:
dist_vetor = pdist(X, metric='mahalanobis', VI=inv)

In [46]:
dist_sem_diag = dist_vetor[dist_vetor > 0]

TAU = np.quantile(dist_sem_diag, 0.25)

In [59]:
THETA = 1.0

W_vector = np.exp(-THETA * dist_vetor**2)

W_vector[dist_vetor > TAU] = 0.0

W = squareform(W_vector)

W = 0.5 * (W + W.T)

In [ ]:
degree = W.sum(axis=1)

L = -W

np.fill_diagonal(L, degree)

L

array([[ 2.61630800e+00,  0.00000000e+00, -4.30944147e-14, ...,
        -5.53579139e-16, -6.29665610e-16,  0.00000000e+00],
       [ 0.00000000e+00,  1.87935870e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-4.30944147e-14,  0.00000000e+00,  1.14326994e+01, ...,
        -1.06538873e-14, -2.65669243e-06,  0.00000000e+00],
       ...,
       [-5.53579139e-16,  0.00000000e+00, -1.06538873e-14, ...,
         9.30217382e-01, -1.48131408e-14,  0.00000000e+00],
       [-6.29665610e-16,  0.00000000e+00, -2.65669243e-06, ...,
        -1.48131408e-14,  1.75884881e+01,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  8.47926944e-14]],
      shape=(10000, 10000))

In [ ]:
class LaplacianFairnessObjective(Objective):
    name = "laplacian_fairness"

    def __init__(
        self,
        L,
        fairness_weight=1.0,
        ce_weight=0.1,
        normalize=False,
        symmetrize=False,
    ):
        self.L = L.float()
        self.fairness_weight = fairness_weight
        self.ce_weight = ce_weight
        self.normalize = normalize
        self.symmetrize = symmetrize

    def __call__(self, logits, y_true, sensitive_attr):
        """
        Calcula a perda laplaciana:

            tr(F^T L F)

        onde F pode ser:
            - logits diretamente;
            - probabilidades, se use_softmax=True.
        """

        F_scores = logits.float()

        if F_scores.ndim == 1:
            F_scores = F_scores.view(-1, 1)

        L = self.L.to(F_scores.device)

        if self.symmetrize:
            L = 0.5 * (L + L.T)

        if L.shape[0] != F_scores.shape[0]:
            raise ValueError(
                f"L tem shape {L.shape}, mas logits tem shape {F_scores.shape}. "
                "O Laplaciano precisa ser gerado sobre o mesmo conjunto usado no fit."
            )

        fairness = torch.trace(F_scores.T @ L @ F_scores)

        if self.normalize:
            fairness = fairness / F_scores.shape[0]

        ce = F.cross_entropy(logits, y_true)

        return self.fairness_weight * fairness + self.ce_weight * ce